# LSTM Model — Architecture and Training

This notebook defines the LSTM architecture for multi-city carsharing demand
prediction and trains it on 9 European cities (2015 data). The trained encoder
learns universal temporal demand patterns that can transfer to unseen cities.

The model checkpoint is saved for evaluation and fine-tuning in the next notebook.

## 1. Configuration and Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_PATH = Path('../results/lstm/dataset_h3_multicidad.parquet')
RESULTS_DIR = Path('../results/lstm')
FIGURES_DIR = Path('../figures')

CITY_TEST = 'muenchen'
MIN_ACTIVE_PCT = 0.15
WINDOW_SIZE = 24
VAL_DAYS = 7

HIDDEN_SIZE  = 64
NUM_LAYERS   = 2
DROPOUT      = 0.2
BATCH_SIZE   = 512
LEARNING_RATE = 1e-3
EPOCHS       = 50
PATIENCE     = 8

EVALUATE_ONLY = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Test city (transfer): {CITY_TEST}')
print(f'Evaluate only: {EVALUATE_ONLY}')

## 2. Loading the H3 Dataset

In [2]:
df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df.sort_values(['city', 'h3_cell', 'date', 'hour'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Total rows: {len(df):,}')
print(f'Cities: {sorted(df["city"].unique())}')

Total rows: 552,384
Cities: ['amsterdam', 'berlin', 'firenze', 'kobenhavn', 'milano', 'muenchen', 'roma', 'stockholm', 'torino', 'wien']


## 3. Active Cell Filtering

H3 cells with fewer than 15% of hours showing any demand are discarded. These
near-empty cells contribute mostly noise and slow down training without
improving generalisation.

In [3]:
cell_activity = (
    df.groupby(['city', 'h3_cell'])['target_demanda']
    .apply(lambda x: (x > 0).mean())
    .reset_index(name='pct_active')
)

active_cells = cell_activity[cell_activity['pct_active'] >= MIN_ACTIVE_PCT]
print(f'Cells before filter: {len(cell_activity):,}')
print(f'Active cells (>={MIN_ACTIVE_PCT*100:.0f}%): {len(active_cells):,}')
print(active_cells.groupby('city')['h3_cell'].count().to_string())

df = df.merge(active_cells[['city', 'h3_cell']], on=['city', 'h3_cell'], how='inner')
print(f'\nRows after filter: {len(df):,}')

Cells before filter: 471
Active cells (>=15%): 311
city
amsterdam    31
berlin       80
firenze      16
kobenhavn    15
milano       31
muenchen     32
roma         34
stockholm    16
torino       18
wien         38

Rows after filter: 357,168


## 4. Feature Normalization

The demand target is normalised to zero mean and unit variance per H3 cell (z-score).
Temporal features (sine/cosine encodings) are already in [-1, 1] and require no
further scaling.

In [4]:
FEATURE_COLS = [
    'target_demanda',
    'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'is_weekend'
]
N_FEATURES = len(FEATURE_COLS)

cell_stats = (
    df.groupby(['city', 'h3_cell'])['target_demanda']
    .agg(['mean', 'std'])
    .reset_index()
)
cell_stats['std'] = cell_stats['std'].replace(0, 1)

df = df.merge(cell_stats, on=['city', 'h3_cell'], how='left')
df['demand_norm'] = (df['target_demanda'] - df['mean']) / df['std']
df['target_demanda_orig'] = df['target_demanda'].copy()
df['target_demanda']      = df['demand_norm']

print(f'Normalisation complete. demand_norm range: [{df["demand_norm"].min():.2f}, {df["demand_norm"].max():.2f}]')

Normalisation complete. demand_norm range: [-1.67, 14.04]


## 5. Sliding Window Dataset (PyTorch)

Each training sample consists of a 24-hour window of features (input) and the
demand value at hour t+1 (target). The denormalisation parameters (mean, std) are
stored alongside each sample for converting predictions back to the original scale.

In [5]:
class SlidingWindowDataset(Dataset):
    def __init__(self, df_subset, T=24):
        self.T = T
        windows, targets, denorm_params = [], [], []

        for (city, cell), grp in df_subset.groupby(['city', 'h3_cell']):
            grp = grp.sort_values(['date', 'hour'])
            vals = grp[FEATURE_COLS].values.astype(np.float32)
            mu   = grp['mean'].iloc[0]
            sigma = grp['std'].iloc[0]

            for i in range(T, len(vals)):
                windows.append(vals[i-T:i])
                targets.append(vals[i, 0])
                denorm_params.append((mu, sigma))

        self.X = torch.tensor(np.array(windows), dtype=torch.float32)
        self.y = torch.tensor(np.array(targets),  dtype=torch.float32)
        self.denorm = denorm_params

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def denormalize(y_norm, denorm_params):
    result = np.zeros_like(y_norm)
    for i, (mu, sigma) in enumerate(denorm_params):
        result[i] = y_norm[i] * sigma + mu
    return np.clip(result, 0, None)

## 6. Train / Validation / Test Split

- **Train**: 9 cities (2015), all data except the last 7 days per city.
- **Validation**: 9 cities, last 7 days.
- **Test**: München (2016), full period — used for zero-shot evaluation.

In [6]:
df_pretrain = df[df['city'] != CITY_TEST].copy()
df_zeroshot = df[df['city'] == CITY_TEST].copy()

cutoff_dates = (
    df_pretrain.groupby('city')['date']
    .max()
    .apply(lambda d: d - pd.Timedelta(days=VAL_DAYS))
)

train_mask = df_pretrain.apply(
    lambda row: row['date'] <= cutoff_dates[row['city']], axis=1
)

df_train = df_pretrain[train_mask].copy()
df_val   = df_pretrain[~train_mask].copy()

print(f'Train:     {len(df_train):,} rows  ({df_train["city"].nunique()} cities)')
print(f'Val:       {len(df_val):,} rows')
print(f'Test:      {len(df_zeroshot):,} rows  ({CITY_TEST})')

Train:     261,144 rows  (9 cities)
Val:       46,872 rows
Test:      49,152 rows  (muenchen)


In [7]:
print('Building training windows...')
train_ds = SlidingWindowDataset(df_train, T=WINDOW_SIZE)
print(f'  Train windows: {len(train_ds):,}')

print('Building validation windows...')
val_ds = SlidingWindowDataset(df_val, T=WINDOW_SIZE)
print(f'  Val windows:   {len(val_ds):,}')

print('Building test windows (zero-shot)...')
test_ds = SlidingWindowDataset(df_zeroshot, T=WINDOW_SIZE)
print(f'  Test windows:  {len(test_ds):,}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Building training windows...
  Train windows: 254,448
Building validation windows...
  Val windows:   40,176
Building test windows (zero-shot)...
  Test windows:  48,384


## 7. Model Architecture

The LSTM model separates the temporal encoder (two stacked LSTM layers) from the
prediction head (dense layers). This separation enables transfer learning: the
encoder weights can be frozen while fine-tuning only the head on a new city.

In [8]:
class DemandLSTM(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout):
        super().__init__()
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        out, _ = self.encoder(x)
        last    = out[:, -1, :]
        return self.head(last).squeeze(-1)


model = DemandLSTM(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {total_params:,}')

DemandLSTM(
  (encoder): LSTM(6, 64, num_layers=2, batch_first=True, dropout=0.2)
  (head): Sequential(
    (0): Dropout(p=0.2, inplace=False)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): ReLU()
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters: 53,825


## 8. Training Loop with Early Stopping

The model is trained with Huber loss (delta=1.0), which is robust to the large
proportion of zero-demand samples. Training stops after 8 epochs without
improvement in validation loss.

In [ ]:
if not EVALUATE_ONLY:
    criterion  = nn.HuberLoss(delta=1.0)
    optimizer  = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_train_loss += loss.item() * len(y_batch)
        epoch_train_loss /= len(train_ds)

        model.eval()
        epoch_val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                pred = model(X_batch)
                epoch_val_loss += criterion(pred, y_batch).item() * len(y_batch)
        epoch_val_loss /= len(val_ds)

        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)
        scheduler.step(epoch_val_loss)

        if epoch % 5 == 0 or epoch == 1:
            print(f'Epoch {epoch:3d}/{EPOCHS} | Train: {epoch_train_loss:.4f} | Val: {epoch_val_loss:.4f}')

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'Early stopping at epoch {epoch}.')
                break

    model.load_state_dict(best_state)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, RESULTS_DIR / 'lstm_pretrained.pt')
    print(f'\nBest val loss: {best_val_loss:.4f}')
else:
    model.load_state_dict(torch.load(RESULTS_DIR / 'lstm_pretrained.pt', map_location=DEVICE))
    model.eval()
    print('Loaded pretrained checkpoint (training skipped).')

## 9. Learning Curve

In [ ]:
if not EVALUATE_ONLY:
    fig, ax = plt.subplots(figsize=(10, 5.5), facecolor='white')

    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'DejaVu Serif'],
        'font.size': 10,
        'figure.dpi': 150,
    })

    epochs = np.arange(1, len(train_losses) + 1)

    ax.plot(epochs, train_losses, color='#2196F3', linewidth=2, marker='o',
            markersize=5, markerfacecolor='white', markeredgewidth=1.5,
            label='Training loss', zorder=3)
    ax.plot(epochs, val_losses, color='#E65100', linewidth=2, marker='s',
            markersize=5, markerfacecolor='white', markeredgewidth=1.5,
            label='Validation loss', zorder=3)

    best_epoch = np.argmin(val_losses) + 1
    best_val = min(val_losses)
    ax.annotate(f'Best: {best_val:.4f}', xy=(best_epoch, best_val),
                xytext=(best_epoch + 1.5, best_val + 0.003),
                fontsize=10, fontweight='bold', color='#E65100',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                bbox=dict(boxstyle='round,pad=0.3', fc='#FBE9E7', ec='black', alpha=0.9))

    ax.axvline(x=len(train_losses), color='#9E9E9E', linestyle=':', linewidth=1.2, alpha=0.7)

    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Huber Loss', fontsize=11)
    ax.set_title('Learning Curve — LSTM Pretraining', fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks(epochs)
    ax.legend(loc='upper right', fontsize=10, framealpha=0.9, edgecolor='#cccccc', fancybox=False)
    ax.grid(axis='both', alpha=0.25, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/lstm-learning-curve.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print('Learning curve skipped (EVALUATE_ONLY=True).')